# ProtoBind-Diff → CTSF: generate candidate binders, triage with a rigorous pipeline

**What this notebook does**
1. Installs Gero's open-source **ProtoBind-Diff** (a masked discrete-diffusion language model that generates SMILES conditioned on protein sequence).
2. Generates candidate small-molecule binders for **cathepsin F (CTSF)** — the gene validated for exceptional longevity in my MR-GAT study — using **two sequence forms** (full-length canonical vs mature catalytic domain) and compares them.
3. Triages the generated molecules with a **leakage-aware, RDKit-based validation pipeline** (validity, drug-likeness, novelty vs known cathepsin binders, property profiling).

**Why two sequence forms?** CTSF is a papain-family cysteine protease with a ~251-residue propeptide cleaved from the active enzyme; ligands bind the *mature* domain (catalytic Cys at residue 290), but BindingDB — ProtoBind-Diff's training source — typically keys targets by the *canonical* full-length sequence. Running both tests whether conditioning form changes the generated chemistry.

> Set Runtime → Change runtime type → **GPU** before running. ProtoBind-Diff computes ESM-2 embeddings on the fly (needs a GPU).
>
> Model license: CC BY-NC 4.0 (non-commercial). This is a research/portfolio demonstration.

## 1 · Environment check (confirm GPU)

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU — set Runtime > Change runtime type > GPU")
import torch  # noqa
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2 · Clone and install ProtoBind-Diff

In [ ]:
%cd /content
![ -d ProtoBind-Diff] || git clone https://github.com/gero-science/ProtoBind-Diff.git
%cd /content/ProtoBind-Diff
!pip -q install -e .
print("\nInstalled. Checking the CLI entry point:")
!protobind-infer --help | head -40

## 3 · CTSF sequences (both forms)

Canonical CTSF (UniProt **Q9UBX1**, 484 aa) and the mature catalytic domain (residues 273–484, matching the LSBio recombinant construct; contains the catalytic Cys at 290).

In [ ]:
CTSF_FULL = (
    "MAPWLQLLSLLGLLPGAVAAPAQPRAASFQAWGPPSPELLAPTRFALEMFNRGRAAGTRA"
    "VLGLVRGRVRRAGQGSLYSLEATLEEPPCNDPMVCRLPVSKKTLLCSFQVLDELGRHVLL"
    "RKDCGPVDTKVPGAGEPKSAFTQGSAMISSLSQNHPDNRNETFSSVISLLNEDPLSQDLP"
    "VKMASIFKNFVITYNRTYESKEEARWRLSVFVNNMVRAQKIQALDRGTAQYGVTKFSDLT"
    "EEEFRTIYLNTLLRKEPGNKMKQAKSVGDLAPPEWDWRSKGAVTKVKDQGMCGSCWAFSV"
    "TGNVEGQWFLNQGTLLSLSEQELLDCDKMDKACMGGLPSNAYSAIKNLGGLETEDDYSYQ"
    "GHMQSCNFSAEKAKVYINDSVELSQNEQKLAAWLAKRGPISVAINAFGMQFYRHGISRPL"
    "RPLCSPWLIDHAVLLVGYGNRSDVPFWAIKNSWGTDWGEKGYYYLHRGSGACGVNTMASS"
    "AVVD"
)
CTSF_MATURE = CTSF_FULL[272:484]   # aa273-484, the active enzyme
assert len(CTSF_FULL) == 484 and CTSF_FULL[289] == "C", "sequence sanity check"
print("full length:", len(CTSF_FULL), "| mature length:", len(CTSF_MATURE))
print("catalytic motif in mature:", "CGSCWAFS" in CTSF_MATURE)

import os
os.makedirs("/content/ctsf", exist_ok=True)
for name, s in [("ctsf_full", CTSF_FULL), ("ctsf_mature", CTSF_MATURE)]:
    with open(f"/content/ctsf/{name}.fasta", "w") as f:
        f.write(f">{name}\n{s}\n")
print("wrote FASTAs to /content/ctsf/")

## 4 · Generate candidate binders (both forms)

`--n_batches 20 --batch_size 50` → up to 1000 molecules per form. First run downloads the checkpoint from HuggingFace (a few minutes). Reduce batches for a quick test.

In [ ]:
import subprocess, sys

def generate(fasta, out, n_batches=20, batch_size=50, steps=250):
    cmd = ["protobind-infer", "--fasta_file", fasta,
           "--output_dir", "/content/ctsf", "--output", out,
           "--n_batches", str(n_batches), "--batch_size", str(batch_size),
           "--sampling_steps", str(steps)]
    print(">>", " ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-2000:]);  print(p.stderr[-1500:] if p.returncode else "")
    return p.returncode

rc1 = generate("/content/ctsf/ctsf_full.fasta",   "gen_full.txt")
rc2 = generate("/content/ctsf/ctsf_mature.fasta", "gen_mature.txt")
print("\nreturn codes:", rc1, rc2)

## 5 · Triage pipeline (RDKit)

Validity → canonicalise/dedupe → drug-likeness (Lipinski / QED) → novelty vs known cathepsin inhibitors → compare the two sequence forms.

**Adapt the `KNOWN_CATHEPSIN_BINDERS` list** to your real reference set (e.g. the CTSF/cathepsin actives from your BindingDB+ChEMBL affinity repo) for a proper novelty check.

In [ ]:
!pip -q install rdkit
from rdkit import Chem, RDLogger
from rdkit.Chem import QED, Descriptors, AllChem, DataStructs
from rdkit.Chem import Lipinski
RDLogger.DisableLog("rdApp.*")
import pandas as pd, numpy as np

def load(path):
    try:
        with open(path) as f:
            return [l.strip() for l in f if l.strip()]
    except FileNotFoundError:
        print("missing:", path); return []

def triage(smiles_list, label):
    rows = []
    seen = set()
    for smi in smiles_list:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        can = Chem.MolToSmiles(m)
        if can in seen:
            continue
        seen.add(can)
        mw, logp = Descriptors.MolWt(m), Descriptors.MolLogP(m)
        hbd, hba = Lipinski.NumHDonors(m), Lipinski.NumHAcceptors(m)
        lip = (mw <= 500) + (logp <= 5) + (hbd <= 5) + (hba <= 10)
        rows.append(dict(form=label, smiles=can, MW=round(mw,1), logP=round(logp,2),
                         HBD=hbd, HBA=hba, QED=round(QED.qed(m),3),
                         lipinski_pass=int(lip==4)))
    df = pd.DataFrame(rows)
    n_raw = len(smiles_list)
    print(f"[{label}] raw={n_raw}  valid+unique={len(df)}  "
          f"drug-like(QED>=0.5)={ (df.QED>=0.5).sum() if len(df) else 0 }  "
          f"Lipinski-pass={ df.lipinski_pass.sum() if len(df) else 0 }")
    return df

df_full   = triage(load("/content/ctsf/gen_full.txt"),   "full")
df_mature = triage(load("/content/ctsf/gen_mature.txt"), "mature")
df = pd.concat([df_full, df_mature], ignore_index=True)
df.to_csv("/content/ctsf/ctsf_generated_triaged.csv", index=False)
print("\nsaved /content/ctsf/ctsf_generated_triaged.csv")
df.sort_values("QED", ascending=False).head(15)

### Novelty vs known cathepsin binders (Tanimoto on Morgan fingerprints)

In [ ]:
# TODO: replace with your real cathepsin/CTSF actives (from the BindingDB+ChEMBL affinity repo)
KNOWN_CATHEPSIN_BINDERS = [
    "O=C(N[C@@H](Cc1ccccc1)C(=O)N1CCCC1)C1CCCCC1",  # placeholder peptidomimetic
    "CC(C)C[C@H](NC(=O)OCc1ccccc1)C(=O)N[C@@H](CC(C)C)C=O",  # placeholder
]
def fp(smi):
    m = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048) if m else None
known_fps = [f for f in (fp(s) for s in KNOWN_CATHEPSIN_BINDERS) if f is not None]

def max_sim_to_known(smi):
    f = fp(smi)
    if f is None or not known_fps:
        return np.nan
    return max(DataStructs.TanimotoSimilarity(f, kf) for kf in known_fps)

if len(df):
    df["max_tanimoto_to_known"] = df.smiles.map(max_sim_to_known)
    df["novel(<0.4)"] = (df.max_tanimoto_to_known < 0.4).astype(int)
    print("novel (Tanimoto<0.4 vs known):", int(df["novel(<0.4)"].sum()), "/", len(df))
    # final shortlist: valid, drug-like, novel
    shortlist = df[(df.QED>=0.5) & (df.lipinski_pass==1) & (df["novel(<0.4)"]==1)]
    shortlist = shortlist.sort_values("QED", ascending=False)
    shortlist.to_csv("/content/ctsf/ctsf_shortlist.csv", index=False)
    print("shortlist (drug-like + novel):", len(shortlist), "-> /content/ctsf/ctsf_shortlist.csv")
    display(shortlist.head(20))

## 6 · Compare the two sequence forms

Does conditioning on the full-length vs mature sequence change the generated chemistry (validity rate, drug-likeness, property distributions, overlap)? This comparison is the methodological contribution — it tests whether sequence form matters for sequence-conditioned generation.

In [ ]:
if len(df):
    summ = df.groupby("form").agg(
        n=("smiles","size"),
        valid_unique=("smiles","nunique"),
        mean_QED=("QED","mean"),
        lipinski_rate=("lipinski_pass","mean"),
        mean_MW=("MW","mean"),
        mean_logP=("logP","mean"),
    ).round(3)
    print(summ.to_string())
    # chemical overlap between the two forms
    sf, sm = set(df_full.smiles), set(df_mature.smiles)
    inter = len(sf & sm); union = len(sf | sm)
    print(f"\nshared molecules between forms: {inter}  (Jaccard {inter/max(union,1):.3f})")

## 7 · Notes for write-up

- **Honest framing:** the generation is Gero's model; the *contribution here is the rigorous triage + the sequence-form comparison*. Don't claim to have built the generator.
- **Validity rate** is the first quality signal — low validity would flag a sequence-form or setup problem.
- **Novelty** depends entirely on the reference set — swap in your real CTSF/cathepsin actives from the affinity repo.
- **Limitation:** generated ≠ validated binders; these are *candidates*. True affinity needs docking/Boltz-1 (as in the ProtoBind-Diff paper) or experimental assay. State this plainly.